In [25]:
#!pip install llama-index-embeddings-huggingface

In [1]:
def initialize_llm(provider: str = "openai", temperature: float = 0.3):
    """
    Initialize and return LLM instance based on provider.
    """
    load_dotenv(override=True)
    
    if provider.lower() == "openai":
        from llama_index.llms.openai import OpenAI
        api_key = os.getenv("OPENAI_API_KEY")
        return OpenAI(model="gpt-4o-mini", temperature=temperature, api_key=api_key)
    
    elif provider.lower() == "gemini":
        from llama_index.llms.gemini import Gemini
        api_key = os.getenv("GEMINI_API_KEY")
        return Gemini(model="gemini-1.5-flash", api_key=api_key, temperature=temperature)
    
    elif provider.lower() == "ollama":
        from llama_index.llms.ollama import Ollama
        return Ollama(model="llama3.2", temperature=temperature, request_timeout=120.0)
    
    elif provider.lower() == "groq":
        from llama_index.llms.groq import Groq
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise EnvironmentError("GROQ_API_KEY not found in environment variables")
        # llama-3.3-70b-versatile is currently one of their best/fastest models
        return Groq(model="llama-3.3-70b-versatile", api_key=api_key, temperature=temperature)
    
    else:
        raise ValueError(f"Unknown provider: {provider}. Choose 'openai', 'gemini', 'ollama', or 'groq'")

In [2]:
import pandas as pd
import csv,os
from dotenv import load_dotenv
import openai
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
LLM_PROVIDER = "groq"  # Change to "gemini" to use Gemini, "ollama" for Ollama, or "openai" for OpenAI
input_folder = r"G:\My Drive\python\llm\udemy_llm\llm_engineering\cte\knowledge-base"

load_dotenv(override=True)

os.environ['HF_TOKEN'] = os.getenv('HUGGINGFACE_API_KEY')

df1 = pd.read_csv(input_folder +'/aact_multiple_sponsors.csv', encoding='utf-8', sep='|', 
                 quoting=csv.QUOTE_MINIMAL, quotechar='"', 
                 on_bad_lines='skip'  # Or use 'warn' to log malformed lines instead 
                )

In [3]:
from sqlalchemy import create_engine
engine = create_engine("sqlite:///:memory:")

In [4]:
df1.to_sql("aact_multiple_sponsors", engine, index=False)

210859

In [7]:

from llama_index.llms.openai import OpenAI
from llama_index.core import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine

# create a new Chat with OpenAI
#llm = OpenAI(temperature=0.7, model_name=LLM_MODEL_ID)
llm =initialize_llm(provider=LLM_PROVIDER, temperature=0.7)
sql_database = SQLDatabase(engine)



In [8]:
#Query-Time Retrieval of Tables for Text-to-SQL

from llama_index.core.indices.struct_store.sql_query import (
    SQLTableRetrieverQueryEngine,
)
from llama_index.core.objects import (
    SQLTableNodeMapping,
    ObjectIndex,
    SQLTableSchema,
)
from llama_index.core import VectorStoreIndex

# manually set context text
tb_aact_multiple_sponsors = (
    "This table gives information regarding the clinical trials that are uniquely identified by NCT_ID"
    " initiated by sponsors .\nThis tablr has basic informaion about study, including study title,"
    " date study registered with ClinicalTrials.gov, date results first posted to ClinicalTrials.gov,"
    "dates for study start and completion, phase of study, enrollment status, planned or actual enrollment, number of study arms/groups, etc."
    " It also has Name(s) of the disease(s) or condition(s) studied in the clinical study, or the focus of the clinical study."
)
# set Logging to DEBUG for more detailed outputs
table_node_mapping = SQLTableNodeMapping(sql_database)
table_schema_objs = [
    (SQLTableSchema(table_name="aact_multiple_sponsors" , context_str=tb_aact_multiple_sponsors))
]  # add a SQLTableSchema for each table

obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_mapping,
    VectorStoreIndex,
    #embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
    embed_model=HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_ID),
)
query_engine = SQLTableRetrieverQueryEngine(
    sql_database, obj_index.as_retriever(similarity_top_k=1) ,llm = llm
)

In [9]:
response = query_engine.query("What is the total count of the table?")
print(response)

The total count of the table is 210,859.


In [21]:
response = query_engine.query("top 5 sponsors with the most studies?")
response.metadata["result"]

[('Assiut University', 2773),
 ('Cairo University', 2661),
 ('Riphah International University', 1632),
 ('Assistance Publique - Hôpitaux de Paris', 1530),
 ('Mayo Clinic', 1100)]

In [11]:
response = query_engine.query("Give me the count of all studies for Novartis by Overall Status")
response.metadata["result"]

[('ACTIVE_NOT_RECRUITING', 1), ('COMPLETED', 17)]

In [12]:
response = query_engine.query("What is the status of studyid JZP598-302 and how many patients were planned to be recruited")
for n in response.metadata["result"]:
    print(n)

('RECRUITING', 286)


In [13]:
print(type(response.metadata["result"]))

<class 'list'>


In [14]:
# # Define the query function
# def text2sql_query(user_query):
#     try:
#         response = query_engine.query(user_query)
#         return str(response)
#     except Exception as e:
#         return f"Error: {str(e)}"

import gradio as gr
with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask about clinical trials...")
    clear = gr.Button("Clear Chat")

    history = []

    def respond(message):
        response = query_engine.query(message)
        history.append([message, str(response)])
        print(history)
        return "", history

    msg.submit(respond, inputs=msg, outputs=[msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch()

C:\Users\nitin\AppData\Local\Temp\ipykernel_15388\3814711296.py:11: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import os
import csv
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from sqlalchemy import create_engine

from llama_index.core import SQLDatabase, VectorStoreIndex
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine
from llama_index.core.objects import SQLTableNodeMapping, SQLTableSchema, ObjectIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 🔧 Constants
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_ID = "gpt-4o-mini"
input_folder = r"G:\My Drive\python\llm\udemy_llm\llm_engineering\cte\knowledge-base"
csv_path = os.path.join(input_folder, "aact_multiple_sponsors.csv")

# 🌿 Load environment variables
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HUGGINGFACE_API_KEY")

if not openai_key or not hf_token:
    raise EnvironmentError("Missing required API keys in environment variables.")

os.environ["OPENAI_API_KEY"] = openai_key
os.environ["HF_TOKEN"] = hf_token

# Load CSV
df1 = pd.read_csv(
    csv_path,
    encoding="utf-8",
    sep="|",
    quoting=csv.QUOTE_MINIMAL,
    quotechar='"',
    on_bad_lines='skip',
    low_memory=False
)

# Create SQL engine and table
engine = create_engine("sqlite:///trial_data.db")
df1.to_sql("aact_multiple_sponsors", engine, index=False, if_exists="replace")

# Initialize LLM and query engine
llm = OpenAI(model_name=LLM_MODEL_ID, temperature=0.7)
sql_database = SQLDatabase(engine)

# Provide schema context
context_text = (
    "This table gives information regarding clinical trials uniquely identified by NCT_ID "
    "initiated by sponsors. It includes study title, registration date, results posting date, "
    "start and completion dates, study phase, enrollment status, number of arms/groups, "
    "and disease or condition studied."
)

table_node_mapping = SQLTableNodeMapping(sql_database)
table_schema_objs = [
    SQLTableSchema(table_name="aact_multiple_sponsors", context_str=context_text)
]

obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_mapping,
    VectorStoreIndex,
    embed_model=HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_ID)
)

query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_index.as_retriever(similarity_top_k=1),
    llm=llm
)

def answer_query(user_question):
    response = query_engine.query(user_question)
    return str(response)

# Store conversation history
# chat_history = []

# def answer_query(user_question):
#     # Combine all previous messages into a single prompt
#     context_prompt = ""
#     for user, assistant in chat_history:
#         context_prompt += f"User: {user}\nAssistant: {assistant}\n"
#     context_prompt += f"User: {user_question}\nAssistant:"

#     try:
#         response = query_engine.query(context_prompt)
#         chat_history.append([user_question, str(response)])
#     except Exception as e:
#         chat_history.append([user_question, f"Error: {e}"])
#     return "", chat_history
MAX_HISTORY = 6
SUMMARY_THRESHOLD = 10  # Number of messages before summarizing

chat_history = []

def summarize_history(history):
    summary_prompt = "Summarize this conversation:\n"
    for user, assistant in history:
        summary_prompt += f"User: {user}\nAssistant: {assistant}\n"
    summary_prompt += "\nSummary:"
    summary = llm.complete(summary_prompt)
    return summary.strip()

def answer_query(user_question):
    # Summarize if history is long
    if len(chat_history) > SUMMARY_THRESHOLD:
        summarized = summarize_history(chat_history[:-MAX_HISTORY])
        context_prompt = f"Summary of earlier conversation:\n{summarized}\n\n"
        truncated_history = chat_history[-MAX_HISTORY:]
    else:
        context_prompt = ""
        truncated_history = chat_history

    # Add recent messages
    for user, assistant in truncated_history:
        context_prompt += f"User: {user}\nAssistant: {assistant}\n"
    context_prompt += f"User: {user_question}\nAssistant:"

    try:
        response = query_engine.query(context_prompt)
        chat_history.append([user_question, str(response)])
    except Exception as e:
        chat_history.append([user_question, f"Error: {e}"])
    return "", chat_history

# Build Gradio UI with Chatbot
with gr.Blocks() as demo:
    gr.Markdown("## LlamaIndex SQL Part 2: Table retrieval + Text‑to‑SQL")
    gr.Markdown("Queries schema dynamically to pick relevant tables at query time, if schema size is large")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask about clinical trials...", label="Your question")
    clear = gr.Button("Clear Chat")

    msg.submit(answer_query, inputs=msg, outputs=[msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch()


C:\Users\nitin\AppData\Local\Temp\ipykernel_43048\2272894480.py:136: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [ ]:

import gradio as gr
from gradio import LikeData

MAX_HISTORY = 6
SUMMARY_THRESHOLD = 10  # Number of messages before summarizing
chat_history = []

def summarize_history(history):
    summary_prompt = "Summarize this conversation:\n"
    for user, assistant in history:
        summary_prompt += f"User: {user}\nAssistant: {assistant}\n"
    summary_prompt += "\nSummary:"
    summary = llm.complete(summary_prompt)
    return summary.strip()

# Do not update history here
def answer_query(user_question):
    system_prompt = (
        "System: You are a clinical trial assistant that strictly answers questions using only the available clinical trial data. "
        "Do not generate or assume any information that is not explicitly present in the data. "
        "If a question cannot be answered based on the data, respond with 'No relevant data found.' "
        "If the question does not directly match available data fields, suggest a related or rephrased question based on the closest matching column names.\n\n"
    )


    if len(chat_history) > SUMMARY_THRESHOLD:
        summarized = summarize_history(chat_history[:-MAX_HISTORY])
        context_prompt = system_prompt + f"Summary of earlier conversation:\n{summarized}\n\n"
        truncated_history = chat_history[-MAX_HISTORY:]
    else:
        context_prompt = system_prompt
        truncated_history = chat_history

    for user, assistant in truncated_history:
        context_prompt += f"User: {user}\nAssistant: {assistant}\n"
    context_prompt += f"User: {user_question}\nAssistant:"

    try:
        response = query_engine.query(context_prompt)
        return user_question, str(response)
    except Exception as e:
        return user_question, f"Error: {e}"

def process_input(user_question):
    user_input, response = answer_query(user_question)
    return "", chat_history + [[user_input, response]]

def handle_feedback(data: LikeData):
    if isinstance(data.value, list) and len(data.value) == 2:
        user_input = data.value[0]
        assistant_response = data.value[1]
        if not data.liked:
            print(f"Feedback: not added to history → {assistant_response}")
        else:
            chat_history.append([user_input, assistant_response])
            print("Feedback: added to history")
            

def clear_chat():
    global chat_history
    chat_history = []
    return []

# UI
with gr.Blocks() as demo:
    gr.Markdown("## ClinicalTrials Explorer")
    gr.Markdown("Chat bot to explore clinicaltrials.gov")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask about clinical trials...", label="Your question")
    clear = gr.Button("Clear Chat")

    msg.submit(process_input, inputs=msg, outputs=[msg, chatbot])
    clear.click(clear_chat, None, chatbot)
    chatbot.like(handle_feedback)

demo.launch()


C:\Users\nitin\AppData\Local\Temp\ipykernel_43048\2158680442.py:69: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.
